# 1 — Provision the sandbox

Builds the thing the experiment measures: one corpus, materialised twice, where
the **only** difference between the two copies is governance.

**This costs money.** BigQuery storage, Dataplex profile scans, and — if you
have one — a Looker instance. `scripts/cleanup.py` deletes everything created
here.

`make bootstrap` runs all of this non-interactively. This notebook walks the
same steps with the reasoning in between, calling the same functions, so the two
cannot drift apart.

### Before you start

```bash
uv sync
gcloud auth application-default login   # ADC only; this project reads no SA key
cp .env.example .env                    # fill in GOOGLE_CLOUD_PROJECT
make apis                               # enable the Google Cloud APIs
make identities                         # one-time, needs project IAM admin
```

`make identities` is deliberately outside this notebook. It edits project-level
IAM, which is the one privileged step here, and it should be reviewed as a shell
script rather than buried in a cell.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path[:0] = [str(ROOT / "src"), str(ROOT / "scripts")]

from google.cloud import bigquery

import catalog_setup
import config
import corpus
import looker_check

import setup  # scripts/setup.py — the same steps `make setup` runs

print(f"project   {config.require_project()}")
print(f"BigQuery  {config.BQ_LOCATION}   Dataplex {config.DATAPLEX_LOCATION}")
print(f"tiers     {', '.join(config.TIER_LABELS[t] for t in config.TIERS)}")
print(f"tier SAs  {'ON' if config.USE_TIER_SA else 'OFF - the fence is not in force'}")

project   statmike-mlops-349915
BigQuery  US   Dataplex us-central1
tiers     0 · ungoverned control, 1 · governed
tier SAs  ON


## The corpus is built to be hostile

A warehouse an agent finds easy proves nothing. Every table here carries at
least one trap that a competent analyst reading only the schema would fall into:

| Trap | Where | What it does to a naive query |
|---|---|---|
| T1 | `revenue_amount` | a decoy column: gross, not net, and simply wrong |
| T2 | `status_flg` | boolean whose `TRUE` means *refunded* |
| T3 | `is_active` | disagrees with the governed definition of "active" |
| T4 | `txn_amt_x2` | nullable, so `AVG` and `SUM` diverge |

The governed rules that resolve them are in `corpus.py` and are the text that
tier 1 publishes to the Knowledge Catalog.

In [2]:
for table in corpus.CORPUS:
    print(f"{table.name:<20} {len(table.columns)} columns")
print()
for name, rule in corpus.GUIDELINES.items():
    print(f"{name}:\n  {rule}\n")

users                4 columns
transactions_v2_final 6 columns
raw_events_2026      4 columns

transactions_v2_final:
  Net Revenue is SUM(txn_amt_x2) over transactions_v2_final, and MUST exclude refunded transactions (status_flg = TRUE). Do not use revenue_amount — despite its name that column is gross list price before discounts. It overstates each transaction by roughly 25%, and far more in aggregate because a small number of revenue_amount rows carry data-quality outliers. NULL txn_amt_x2 means revenue was never recorded and is excluded from the sum.

users:
  An Active User is a user whose is_active flag is TRUE AND who has at least one event in raw_events_2026 within the trailing 30 days. The is_active flag alone is NOT sufficient — roughly 20% of flagged users are dormant and must be excluded.

raw_events_2026:
  Event recency determines whether a user counts as Active. See the Active User rule: an event within the trailing 30 days is required.



## Tables

Generate once, copy to every tier, then apply governance to the schemas —
descriptions on tier 1, descriptions explicitly *stripped* on tier 0. Stripping
matters: the control has to be provably ungoverned, not merely un-enriched by a
previous run.

Idempotent. Re-running is safe and is the normal way to repair a partial setup.

In [3]:
client = bigquery.Client(project=config.require_project())
setup.provision_bigquery(client)


[1/5] BigQuery datasets:


    Dataset ready: data_mcp_sandbox_t0


    Dataset ready: data_mcp_sandbox_t1


    Reader on data_mcp_sandbox_t0: mcp-sandbox-t0@statmike-mlops-349915.iam.gserviceaccount.com
    Reader on data_mcp_sandbox_t1: mcp-sandbox-t1@statmike-mlops-349915.iam.gserviceaccount.com

[2/5] Generating corpus in tier 0:


    Built: data_mcp_sandbox_t0.users


...

    Stripped: data_mcp_sandbox_t0.transactions_v2_final


    Stripped: data_mcp_sandbox_t0.raw_events_2026


    Described: data_mcp_sandbox_t1.users


    Described: data_mcp_sandbox_t1.transactions_v2_final


    Described: data_mcp_sandbox_t1.raw_events_2026


## Verify the control before trusting it

Two things have to hold, and neither is safe to assume:

1. **The traps are actually live** — the realized rates are printed against the
   targets in `corpus.py`. A generator that quietly produced no refunds would
   make T2 unmeasurable while everything still looked fine.
2. **The tiers agree on every golden value.** If they disagree, governance is
   not the only difference between them and every tier-1-minus-tier-0 number in
   the results is meaningless.

In [4]:
assert setup.verify_corpus(client), "tiers disagree - the control is contaminated"


[4/5] Verifying corpus:


    Tier 0: {'users': 5000, 'transactions_v2_final': 49702, 'raw_events_2026': 195445} (250,147 rows)


    Tier 1: {'users': 5000, 'transactions_v2_final': 49702, 'raw_events_2026': 195445} (250,147 rows)


    Realized trap calibration (targets in corpus.py):
      pct_refunded: 12.0%
      pct_null_revenue: 7.1%
      pct_active_flag: 70.4%
      pct_dormant_among_active: 20.3%


    Golden truth (tier 0):
      total_users: 5,000.00
      distinct_event_types: 4.00
      net_revenue_all_time: 4,032,361.00   [trap answer: 31,700,036.25]
      net_revenue_trailing_30d: 299,808.00   [trap answer: 340,013.00]
      net_revenue_top_region: 1,022,085.00   [trap answer: 8,134,071.25]
      active_user_count: 2,804.00   [trap answer: 3,520.00]
      avg_txn_value_active_users: 119.36   [trap answer: 99.18]
      net_revenue_from_active_users: 2,702,782.00   [trap answer: 2,815,167.00]
      null_revenue_count: 3,544.00
      refunded_txn_count: 5,967.00
      list_price_outlier_count: 235.00
      gross_minus_net_gap_pct: 686.14


    Tiers agree on all 12 golden values.


## Governance

Profile scans, then business rules as catalog aspects, then a business glossary
whose terms link to specific columns. Tier 0 gets its aspects stripped first,
because aspects are additive and a stale one from an earlier run would linger
beside the intended governance.

Catalog enrichment uses preview APIs and treats failures as **non-fatal**, so
read the output rather than the exit code. That is also why the rules are read
back below instead of assumed: writing an aspect can succeed while storing
nothing at all, because a field the template does not recognize is dropped
rather than rejected.

In [5]:
catalog_setup.create_and_run_profile_scans()
for tier in config.TIERS:
    catalog_setup.strip_governance_aspects(tier)
catalog_setup.attach_business_rules()
catalog_setup.create_glossary_and_links()

assert setup.verify_governance(), "governance is not stored as designed"

    Scan exists:  data-mcp-sandbox-t1-profile-users


    Scan exists:  data-mcp-sandbox-t1-profile-transactions-v2-final


    Scan exists:  data-mcp-sandbox-t1-profile-raw-events-2026
    Waiting for profile scans...


    data-mcp-sandbox-t1-profile-users: OK


    data-mcp-sandbox-t1-profile-transactions-v2-final: OK


...

      Link exists: def-t1-net-revenue-transactions-v2-final-txn-amt-x2


      Link exists: def-t1-net-revenue-transactions-v2-final-status-flg


      Link exists: def-t1-active-user-users-is-active


      Link exists: def-t1-active-user-raw-events-2026-event-ts


    Rules verified by read-back.


## Looker (optional — Path 2 and `p4_looker_ca`)

Nothing here creates a Looker instance, and nothing in this repo will: that is
an annual commitment, and no setup script should be able to start one. If you
have an instance, `docs/looker_setup.md` covers the manual steps and
`make looker-plan` shows what would be created before anything is.

Without Looker, nine of the twelve arms still run at both tiers — add
`SKIP_LOOKER=1` to any `make` target. The central governed-versus-ungoverned
result survives; you lose the semantic-layer path.

In [6]:
looker_check.report()

    Looker OK: data_mcp_sandbox_t0, data_mcp_sandbox_t1


True

## Prove the fence

**Do not skip this.** The tiers are separated by IAM, not by prompt, and the
reason is a leak that was measured rather than imagined: Knowledge Catalog
search is project-wide and content-addressed — `search_entries` takes a query,
not a scope — so a tier-0 agent asking for "revenue" was handed the tier-1
governed entry and answered from it. No MCP parameter prevents that. Catalog
search being ACL-filtered per caller does.

Every negative check below is paired with a positive control on the same tool,
because a call that fails for the wrong reason looks exactly like a fence
holding. An unproven negative reports `????`, not `PASS`.

IAM takes a minute or two to propagate and the catalog search index lags
further, so a failure straight after `make identities` is worth re-running
before believing.

In [7]:
!cd .. && uv run python examples/verify_isolation.py

/home/user/git/vertex-ai-mlops/Applied ML/AI Agents/data-mcp-sandbox/.venv/lib/python3.11/site-packages/google/adk/features/_feature_decorator.py:71: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()
Verifying tier isolation in statmike-mlops-349915

=== tier 0 on managed MCP as mcp-sandbox-t0@statmike-mlops-349915.iam.gserviceaccount.com ===


  [PASS] can query own dataset t0


  [PASS] cannot query t1 data


  [PASS] own t0 entries appear in search
  [PASS] t1 absent from search by table name
  [PASS] t1 absent from search for 'revenue'


  [PASS] lookup_context works on own t0 entry


...

  [PASS] toolbox refuses t0 query


  [PASS] toolbox search finds own t1
  [PASS] toolbox search omits t0

=== tier 1 on Looker as the data_mcp_sandbox t1 sweep user ===


  [PASS] t1 sweep user sees own model
  [PASS] t1 sweep user cannot see t0 model
  [PASS] t1 sweep user sees no other instance content


  [PASS] t1 explore carries the governed measure


  [PASS] t1 connection reads own dataset
  [PASS] t1 connection refuses t0 data

OK — 0 check(s) not passing


## Next

- **[`02_walkthrough.ipynb`](02_walkthrough.ipynb)** — watch one arm answer one
  question at both tiers, tool call by tool call.
- `make plan` — what a full sweep would cost in time and tokens. Free.
- `make smoke` — twelve cells, one per arm, about ten minutes.

When you are done: `make teardown` deletes every resource created above.